# FANCY TITLE HERE

**42186 Model-based Machine Learning — final project, group submission.**

This notebook is the single source of truth for our model comparison. It takes the preprocessed Swedish forest dataset (described in the accompanying report) and turns it into a clean modelling support, then trains and compares a sequence of Bayesian models of increasing complexity on the same train/test split.

### What we are predicting

For each $12.5\,\mathrm{m} \times 12.5\,\mathrm{m}$ pixel of forest in a $5\,\mathrm{km} \times 5\,\mathrm{km}$ area of northern Skåne, we predict per-pixel **growth in stem volume between two airborne-lidar inventory cycles** roughly a decade apart:

$$
\Delta V \;=\; \texttt{volume\_t2} \;-\; \texttt{volume\_t1} \quad [\mathrm{m^3/ha}].
$$

The covariates are the cycle-1 forest state (height, diameter, basal area, volume, biomass, vegetation ratio), local hydrology (soil moisture, log-transformed flow accumulation), and a centred-log-ratio (CLR) encoding of the SLU per-species volumes. The PGM and feature rationale are in §3 of the report.

### How this notebook is laid out

| § | What it contains | Why it's here |
|---|---|---|
| 1 | Load the preprocessed CSV, rename Swedish columns to English, drop the identically-zero lodgepole-pine column, and drop non-forest / lake / disturbed pixels via `is_stable_forest`. | Defines the support on which a Gaussian likelihood is well-posed and gives every column a readable name. |
| 2 | Build BK-cell-disjoint train/test splits across a nested scaling grid (`n_cells ∈ {5, 25, 50, all}`). | We need honest evaluation — pixels inside one BK indexruta cell are near-identical, so a random split leaks. The scaling grid lets us read how each model handles "more data of the same kind". |
| 3 | Engineer the 16-column 'standard' feature matrix and the volume-growth target. | Decouples feature choice from any single model. |
| 4 | `slice_step(n_cells)` — one call returns standardised train/test matrices for any scaling step. | Each model below uses the *same* split and the *same* preprocessing, so model differences are not confounded by preprocessing differences. |
| 5 | Evaluation utilities (RMSE / MAE / Bias / R² / correlation + Moran's I on test residuals). | Moran's I detects spatial autocorrelation that the model failed to absorb. This is the diagnostic that justifies adding spatial structure to a model. |
| 6 | Point-estimate baselines: **OLS** (MLE of §7.1's likelihood) and **HistGradientBoosting** (a flexible nonlinear tree ensemble), both across the scaling grid. | OLS is the linear point-accuracy floor; GBM is the nonlinear point-accuracy ceiling. §7's Bayesian models have to justify themselves against both. |
| 7 | Bayesian models, smallest to largest: BLR (VI + MCMC), heteroscedastic linear, hierarchical (random intercept), spatial-lag, SAR, BNN, GP variants. | This is the comparison the report's "Models" section discusses. Each one inherits §4–§5 unchanged. |
| 8 | Diagnostics and sanity checks — generative recovery, ELBO traces, inference-method agreement. | **The course brief flags these explicitly.** |

In [ ]:
from __future__ import annotations
from pathlib import Path
import json
import urllib.request

import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import r2_score

SEED = 42
np.random.seed(SEED)

# Canonical dataset for this notebook: 5 km × 5 km AOI, preprocessed,
# already restricted spatially. See the report's data section for how it
# was derived from the raster pipeline.
CSV_PATH = Path("out_5km_idx_preprocessed.csv")
CSV_URL = (
    "https://github.com/Somon8/mbml-forest-pipeline/"
    "releases/download/v2.0-data-5km/out_5km_idx_preprocessed.csv"
)
CACHE_DIR = Path("cache")
CACHE_DIR.mkdir(exist_ok=True)
SPLITS_JSON = CACHE_DIR / "splits_5km.json"

if not CSV_PATH.exists():
    print(f"CSV missing — downloading from {CSV_URL} ...")
    urllib.request.urlretrieve(CSV_URL, CSV_PATH)
    print(f"  → wrote {CSV_PATH} ({CSV_PATH.stat().st_size/1e6:.0f} MB)")

## 1. Load and filter

The CSV (`out_5km_idx_preprocessed.csv`) is the preprocessed Swedish-forest dataset at $12.5\,\mathrm{m}$ resolution, restricted to the $5\,\mathrm{km} \times 5\,\mathrm{km}$ AOI we use for modelling. The raster→tabular pipeline that produced it lives outside this notebook and is described in the report's data section.

One filter is applied here:

- **`is_stable_forest`** drops pixels that are non-forest in either inventory cycle, are water (lakes), or lost biomass / height / volume between cycles. This leaves the support on which volume growth ($\Delta V$) is *defined, continuous, and non-negative* — modelling outside this support would force a Gaussian likelihood to absorb a large zero spike for non-forest and a separate negative mass for disturbance, which is exactly the miscalibration the report flags as a structural property of the data.

In [ ]:
# Swedish raster column names → readable English. Pipeline outputs and other
# notebooks (04, 06) still use the Swedish names; the rename happens here at
# load time so this notebook is self-contained for a reviewer.
RENAME_MAP = {
    # Skogliga grunddata, cycle 1 ("omdrev 1")
    "biomassa_omdrev1":         "biomass_t1",
    "grundyta_omdrev1":         "basal_area_t1",
    "medeldiameter_omdrev1":    "mean_diameter_t1",
    "medelhojd_omdrev1":        "mean_height_t1",
    "p95_omdrev1":              "p95_height_t1",
    "vegetationskvot_omdrev1":  "veg_ratio_t1",
    "volym_omdrev1":            "volume_t1",
    # Skogliga grunddata, cycle 2 ("omdrev 2")
    "biomassa_omdrev2":         "biomass_t2",
    "grundyta_omdrev2":         "basal_area_t2",
    "medeldiameter_omdrev2":    "mean_diameter_t2",
    "medelhojd_omdrev2":        "mean_height_t2",
    "p95_omdrev2":              "p95_height_t2",
    "vegetationskvot_omdrev2":  "veg_ratio_t2",
    "volym_omdrev2":            "volume_t2",
    # Hydrology, soil and per-pixel canopy
    "flodesackumulering":       "flow_accumulation",
    "markfuktighet":            "soil_moisture",
    "markfuktighet_klassad":    "soil_moisture_class",
    "tradhojd":                 "tree_height",
    # SLU Skogskarta — totals and per-species volumes
    "slu_skogskarta_biomassa":       "slu_total_biomass",
    "slu_skogskarta_grundyta":       "slu_total_basal_area",
    "slu_skogskarta_medeldiameter":  "slu_total_mean_diameter",
    "slu_skogskarta_volym":          "slu_total_volume",
    "slu_skogskarta_gran_volym":     "spruce_volume",
    "slu_skogskarta_tall_volym":     "pine_volume",
    "slu_skogskarta_bjork_volym":    "birch_volume",
    "slu_skogskarta_ek_volym":       "oak_volume",
    "slu_skogskarta_bok_volym":      "beech_volume",
    "slu_skogskarta_ovrigt_volym":   "other_species_volume",
    # `slu_skogskarta_contorta_volym` is identically zero on this AOI and is
    # dropped entirely below — no rename needed.
}

df_raw = pd.read_csv(CSV_PATH, low_memory=False)
df_raw = df_raw.rename(columns=RENAME_MAP)
df_raw = df_raw.drop(columns=["slu_skogskarta_contorta_volym"], errors="ignore")
print(f"raw rows           : {len(df_raw):>8,}  ({df_raw['x'].nunique()} × {df_raw['y'].nunique()} grid)")

df = df_raw[df_raw["is_stable_forest"].astype(bool)].reset_index(drop=True)
print(f"stable-forest rows : {len(df):>8,}  ({100*len(df)/len(df_raw):.1f}% of grid)")
print(f"unique BK cells    : {df['BK'].nunique():>8,}")

**Reading the output.** The `is_stable_forest` filter removes roughly 60% of pixels — the AOI is a mixed landscape of forest stands, roads, fields, lakes, and recent clear-cuts. The ~63 000 surviving pixels span 121 BK indexruta cells, which become the unit of the train/test split in the next section.

## 2. BK-cell-disjoint splits on a nested scaling grid

**Why splitting by BK indexruta cell.** Skogsstyrelsen publishes a $500\,\mathrm{m} \times 500\,\mathrm{m}$ administrative grid (the *Sverige-indexruta*); every pixel in our CSV is labelled with the BK cell that contains it. Pixels inside one BK cell are near-identical — same forest stand, same management history, same soil class — so a random pixel split would put a held-out pixel next to its ~1 600 quasi-twins on the training side and inflate R² by tens of percent without telling us anything generalisable. **Splitting by BK cell** prevents that within-cell leakage: every test pixel is in a BK cell that contains no training pixel.

**What it does and does not solve.** BK-disjoint splits kill within-cell leakage. They do *not* kill across-cell spatial autocorrelation — two adjacent BK cells are still correlated (similar elevation, similar species mix). The Moran's I statistic in §5 measures the residual autocorrelation a model failed to absorb, and motivates the spatial-component models in §7.

**How train and test are built.** From the 121 BK cells in the AOI:

- **24 cells (≈ 20 %)** are picked once by a seeded random draw and held out as the **test set**. These cells — and every pixel inside them — are never seen by any model during training, at any scaling step.
- **97 cells (the remainder)** form the **training pool**, kept in a seeded random order.

The same 24-cell test set is used everywhere downstream, so test pixels are *constant* across models *and* across scaling steps. RMSEs are therefore directly comparable.

**Why a *nested* scaling grid.** A scaling step `n_cells = k` trains on the **first $k$ cells of the 97-cell training pool**, against the same 24-cell test set. Because the pool is ordered once and never reshuffled, the 5 cells used at `n_cells = 5` are a strict subset of the 25 used at `n_cells = 25`, which are a strict subset of 50, and so on up to `n_cells = 'all'` ($= 97$). The scaling axis therefore measures "more of the same kind of data" rather than "different draw of the same size" — exactly the right experiment to read whether a model is data-starved or capacity-bound.

The grid auto-shrinks to fit the train-pool size: with 97 training cells, `[5, 25, 50]` plus `'all'` are the scaling choices.

In [ ]:
TEST_FRAC = 0.20


def build_splits(df: pd.DataFrame, test_frac: float, seed: int) -> dict:
    rng = np.random.default_rng(seed)
    all_bk = sorted(df["BK"].unique().tolist())
    rng.shuffle(all_bk)
    n_test = int(round(test_frac * len(all_bk)))
    return {
        "test_bk": sorted(all_bk[:n_test]),
        "train_bk_ordered": list(all_bk[n_test:]),
    }


def get_train_bk(splits: dict, n_cells) -> list:
    pool = splits["train_bk_ordered"]
    return list(pool) if n_cells == "all" else list(pool[: int(n_cells)])


if SPLITS_JSON.exists():
    splits = json.loads(SPLITS_JSON.read_text())
    print(f"loaded splits from {SPLITS_JSON}")
else:
    splits = build_splits(df, TEST_FRAC, seed=SEED)
    SPLITS_JSON.write_text(json.dumps(splits))
    print(f"built and saved splits → {SPLITS_JSON}")

# Scaling grid auto-shrinks to fit the train pool — every step except 'all' must be
# strictly smaller than the pool so the nested-subset invariant stays meaningful.
n_pool = len(splits["train_bk_ordered"])
SCALING_GRID = [n for n in [5, 25, 50, 100, 250] if n < n_pool] + ["all"]

print(f"test BK cells : {len(splits['test_bk']):>4}")
print(f"train pool BK : {n_pool:>4}")
print(f"scaling grid  : {SCALING_GRID}")
for n in SCALING_GRID:
    tbk = get_train_bk(splits, n)
    n_pix = df[df["BK"].isin(set(tbk))].shape[0]
    print(f"  n_cells={str(n):>4}  → {len(tbk):>4} BK cells, {n_pix:>7,} pixels")

In [ ]:
# nesting + disjointness sanity checks
subsets = [set(get_train_bk(splits, n)) for n in SCALING_GRID]
for i in range(len(subsets) - 1):
    assert subsets[i].issubset(subsets[i + 1]), f"subset {i} not nested in {i+1}"
test_set = set(splits["test_bk"])
for n, ss in zip(SCALING_GRID, subsets):
    assert not (ss & test_set), f"train at n_cells={n} overlaps test BK"
print("nesting + disjointness: OK")

**Reading the output.** 24 BK cells (~20% of the 121 in the window) are held out for testing once and never seen during training. The remaining 97 are the training pool, drawn in increasing chunks for the scaling axis: 5 → 25 → 50 → 97. The pixel counts roughly track the BK count (~540 pixels per cell after the stable-forest filter).

## 3. Feature matrix and target

**Target — volume growth.** $\Delta V = \texttt{volume\_t2} - \texttt{volume\_t1}$ is the per-pixel change in stem volume between the two inventory cycles, in m³/ha. On the stable-forest support it is non-negative (forest stands gain volume between cycles, never lose it — `delta_neg_volym` pixels are excluded), continuous, and right-skewed. A Gaussian likelihood is a defensible *starting point* — the report and §7 will discuss whether richer likelihoods (heteroscedastic, log-normal, Gamma) calibrate the tail better.

**Features — 16 columns, three logical blocks:**

1. **Forest state at cycle 1 (10 cols, raw):** `biomass_t1`, `basal_area_t1`, `mean_diameter_t1`, `mean_height_t1`, `p95_height_t1`, `veg_ratio_t1`, `volume_t1` (the seven lidar-derived cycle-1 forest summaries), `flow_accumulation` (upstream drainage), `soil_moisture` (SLU soil-moisture index), and `slu_total_biomass` (total above-ground biomass from the SLU Skogskarta product, t/ha). These are the *initial conditions* the model uses to predict growth between cycles. Cycle-2 columns are deliberately **excluded** — they would leak the target.
2. **Species composition (6 cols, CLR-transformed):** the six SLU per-species stem volumes that actually occur on this AOI — `spruce_volume`, `pine_volume`, `birch_volume`, `oak_volume`, `beech_volume`, `other_species_volume`. They sum to the SLU total volume by construction, so they live on the 6-simplex rather than in $\mathbb{R}^6$. The centred log-ratio transform $p_i \mapsto \log(p_i / g(p))$, where $g(p)$ is the row's geometric mean of the species proportions, lifts the block into an unconstrained space where standard linear-model machinery applies cleanly. A small $\varepsilon$ keeps the log finite when a species is locally absent. Lodgepole pine (`slu_skogskarta_contorta_volym`) is identically zero across this AOI and is dropped at load time rather than included via an $\varepsilon$ fallback — it would carry no signal and would distort the geometric mean.

**A subtle leakage point.** `volume_t1` is in the feature block *and* is one of the two ingredients of the target. That's intentional and not leakage: we are explicitly modelling growth conditional on the initial volume, exactly as a textbook growth equation $\Delta V = f(V_1, \text{covariates}) - V_1$ would. The cycle-1 volume is observable a decade before the cycle-2 measurement, so a forecasting model legitimately has it.

**Standardisation happens later.** Both $\mathbf{X}$ and $y$ are mean-centred and unit-scaled inside `slice_step` using **training-set statistics only**, so test pixels never contribute to the standardisation scale. The unstandardised arrays are kept here as the source of truth.

In [ ]:
BASE_COLS = [
    "biomass_t1", "basal_area_t1", "mean_diameter_t1",
    "mean_height_t1", "p95_height_t1", "veg_ratio_t1",
    "volume_t1", "flow_accumulation", "soil_moisture",
    "slu_total_biomass",
]
SPECIES_COLS = [
    "spruce_volume", "pine_volume", "birch_volume",
    "oak_volume", "beech_volume", "other_species_volume",
]


def clr_transform(values: np.ndarray, eps: float = 1e-9) -> np.ndarray:
    row_sums = values.sum(axis=1, keepdims=True)
    nz = row_sums.ravel() > 0
    props = np.where(row_sums > 0, values / np.where(row_sums > 0, row_sums, 1.0), 0.0)
    props_safe = np.where(props <= 0, eps, props)
    log_props = np.log(props_safe)
    clr = log_props - log_props.mean(axis=1, keepdims=True)
    clr[~nz] = 0.0
    return clr


X_base = df[BASE_COLS].to_numpy(float)
X_clr = clr_transform(df[SPECIES_COLS].to_numpy(float))
X_all = np.hstack([X_base, X_clr])
FEATURE_NAMES = BASE_COLS + [c + "_clr" for c in SPECIES_COLS]

# Target: per-pixel change in stem volume between inventory cycles (m³/ha).
# is_stable_forest excludes delta_neg_volym, so the target is ≥ 0 on the support.
y_all = (df["volume_t2"] - df["volume_t1"]).to_numpy(float)
coords_all = df[["x", "y"]].to_numpy(float)
bk_all = df["BK"].to_numpy()

print(f"X_all   : {X_all.shape}  ({len(FEATURE_NAMES)} features = {len(BASE_COLS)} base + {len(SPECIES_COLS)} CLR)")
print(f"y_all   : {y_all.shape}  mean={y_all.mean():.2f} m³/ha  std={y_all.std():.2f} m³/ha")
print(f"coords  : {coords_all.shape}")

**Reading the output.** Design matrix is $\sim\!63\,000 \times 17$. 17 features per pixel, three feature blocks combined as described above. The target's mean ($\sim\!52\,\mathrm{m^3/ha}$) and standard deviation ($\sim\!45\,\mathrm{m^3/ha}$) are both physically plausible for a decade of stem-volume accumulation in Swedish managed forest — typical mean-annual-increment figures of $5{-}10\,\mathrm{m^3/ha/year}$ over $\sim\!10$ years bracket the observed mean. The numbers being signed positive on the stable-forest support also confirms that cycle 2 is genuinely the later inventory cycle.

## 4. `slice_step` — one call returns one scaling step's data

Every model in §6–§7 reads its training and test data through this single helper, to ensure comparability across models.

`slice_step(n_cells)` returns a dict containing:

- `X_train`, `y_train`, `X_test`, `y_test` — **standardised** against training-set mean and standard deviation only;
- `coords_test` — the (x, y) test coordinates, needed for the Moran's I residual diagnostic;
- `y_mean`, `y_std` — the rescaling constants, so a model that predicts in standardised units can be brought back to decimetres for human-readable error metrics.

In [ ]:
def slice_step(n_cells):
    train_bk = set(get_train_bk(splits, n_cells))
    test_bk = set(splits["test_bk"])

    tr = np.isin(bk_all, list(train_bk))
    te = np.isin(bk_all, list(test_bk))

    Xtr, ytr = X_all[tr], y_all[tr]
    Xte, yte = X_all[te], y_all[te]
    coords_te = coords_all[te]

    x_mean = Xtr.mean(axis=0)
    x_std = Xtr.std(axis=0) + 1e-8
    y_mean, y_std = float(ytr.mean()), float(ytr.std() + 1e-8)

    Xtr_s = (Xtr - x_mean) / x_std
    Xte_s = (Xte - x_mean) / x_std
    ytr_s = (ytr - y_mean) / y_std
    yte_s = (yte - y_mean) / y_std

    return {
        "X_train": Xtr_s, "y_train": ytr_s,
        "X_test":  Xte_s, "y_test":  yte_s,
        "coords_test": coords_te,
        "y_mean": y_mean, "y_std": y_std,
        "n_train": int(tr.sum()), "n_test": int(te.sum()),
    }

## 5. Evaluation utilities

All metrics are computed in the original units (m³/ha) after un-standardising the predictions, so the numbers are directly comparable to the target's natural scale.

**Point-prediction metrics.** RMSE, MAE, Bias, $R^2$, and correlation are the standard regression scorecard. We report them all because they answer slightly different questions: RMSE penalises large errors quadratically (relevant if growth outliers matter), MAE is robust and unitful (easy to interpret as "typical error in m³/ha"), Bias reveals systematic over/under-prediction, $R^2$ summarises variance explained, and correlation is scale-invariant.

**Moran's I on test residuals.** The diagnostic that justifies most of the spatial structure in §7. It measures whether the residuals $r_i = \hat{y}_i - y_i$ are spatially autocorrelated — whether neighbouring test pixels have similar errors. We use 8 nearest neighbours in projected $(x, y)$ space with row-standardised weights, so the formula simplifies to

$$ I \;=\; \frac{\sum_i z_i \cdot \overline{z_{\,N(i)}}}{\sum_i z_i^2}, \qquad z_i = r_i - \bar r, $$

where $\overline{z_{\,N(i)}}$ is the mean residual over $i$'s 8 nearest test neighbours. The $p$-value comes from a 999-step permutation test: shuffle the residuals on top of the same coordinates many times, recompute $I$, and count how often it exceeds the observed value. The implementation is ~25 lines of `numpy` + `sklearn.NearestNeighbors` so the notebook has no `libpysal` / `esda` dependency.

- $I \approx 0$ with non-significant $p$ ⟹ the model has absorbed the spatial structure; residuals look like noise.
- $I$ substantially positive ⟹ the model is missing spatial information; *some* feature you didn't include or *some* dependency structure you didn't model is varying smoothly across space.

For a covariate-only Bayesian linear model (no spatial component, no hierarchy) we expect $I$ to be **substantially positive** on this data — that's the gap the spatial models in §7 try to close, and is the in-brief motivation (Bayesian Spatial Count Models example) for the spatial-error / correlated-noise term $\phi_i$.

In [ ]:
from sklearn.neighbors import NearestNeighbors


def moran_i(values: np.ndarray, coords: np.ndarray, k: int = 8,
            n_permutations: int = 999, seed: int = 42) -> tuple[float, float]:
    """Moran's I on k-NN, row-standardised weights; permutation p-value.

    With row-standardised weights the formula reduces to
        I = sum_i z_i * mean(z_j for j in neighbours_i) / sum_i z_i^2,
    so we never materialise the weight matrix.
    """
    coords = np.asarray(coords)
    values = np.asarray(values).ravel()
    nn = NearestNeighbors(n_neighbors=k + 1).fit(coords)
    _, idx = nn.kneighbors(coords)
    neighbours = idx[:, 1:]  # drop self

    def _i(z: np.ndarray) -> float:
        num = (z * z[neighbours].mean(axis=1)).sum()
        return float(num / (z @ z))

    z_obs = values - values.mean()
    observed = _i(z_obs)

    rng = np.random.default_rng(seed)
    perms = np.fromiter(
        (_i(rng.permutation(z_obs)) for _ in range(n_permutations)),
        dtype=float, count=n_permutations,
    )
    # One-sided permutation p-value (positive autocorrelation is the alternative
    # we expect; libpysal calls this p_sim with this same convention).
    p = (1 + (perms >= observed).sum()) / (1 + n_permutations)
    return observed, float(p)


def evaluate(y_true: np.ndarray, y_pred: np.ndarray, coords: np.ndarray | None = None) -> dict:
    y_true = np.asarray(y_true).ravel()
    y_pred = np.asarray(y_pred).ravel()
    res = y_pred - y_true
    out = {
        "RMSE": float(np.sqrt(np.mean(res ** 2))),
        "MAE":  float(np.mean(np.abs(res))),
        "Bias": float(res.mean()),
        "R2":   float(r2_score(y_true, y_pred)),
        "Corr": float(np.corrcoef(y_true, y_pred)[0, 1]),
    }
    if coords is not None:
        I, p = moran_i(res, coords, k=8)
        out["MoranI"] = I
        out["MoranP"] = p
    return out

## 6. Point-estimate baselines — OLS and GBM across the scaling grid

Two reference points that anchor §7 from below. Both produce point estimates only (no uncertainty), and both share the §4 train/test scaffolding, so §7's Bayesian models can be compared on the *exact* same support.

- **OLS (MLE)** fits the *same linear-Gaussian likelihood* §7.1 will then turn into a full posterior. Any §7 model that does *worse* than OLS in point accuracy is a candidate bug — using a prior should at least match using none.
- **HistGradientBoosting (GBM)** is a flexible nonlinear tree ensemble. It stands in for "what would a state-of-the-art non-Bayesian tabular method do?" with no spatial information beyond the features. Any §7 Bayesian model that closes the OLS → GBM gap on RMSE is showing real capacity gains; any that doesn't has to justify itself on *uncertainty*, *hierarchy*, or *spatial structure* instead.

**What to look for in the output table.**

- **OLS RMSE roughly flat across the scaling grid.** With $\sim\!52\,\mathrm{m^3/ha}$ mean and $\sim\!45\,\mathrm{m^3/ha}$ std on the target, an RMSE in the low-to-mid 30s means the linear model captures around two-thirds of the variance and has hit its capacity ceiling.
- **GBM RMSE noticeably lower than OLS,** and (unlike OLS) actually benefitting from more data — that's the signature of a flexible learner that can use additional rows to fit the nonlinear part of the response surface.
- **Bias close to zero** for both. A systematic bias would signal a standardisation bug.
- **Moran's I substantially positive** for both at every scaling step. Neither point-estimate model has any mechanism to absorb spatial autocorrelation. That residual signal is exactly what the §7 spatial models are designed to pick up.

In [ ]:
BASELINES = [
    ("OLS",  lambda: LinearRegression()),
    ("GBM",  lambda: HistGradientBoostingRegressor(max_iter=300, random_state=SEED)),
]

rows = []
for n_cells in SCALING_GRID:
    s = slice_step(n_cells)
    for name, make in BASELINES:
        est = make().fit(s["X_train"], s["y_train"])
        # back to m³/ha for interpretable units
        yhat = est.predict(s["X_test"]) * s["y_std"] + s["y_mean"]
        ytrue = s["y_test"] * s["y_std"] + s["y_mean"]
        m = evaluate(ytrue, yhat, coords=s["coords_test"])
        rows.append({
            "n_cells": n_cells, "model": name,
            "n_train": s["n_train"], "n_test": s["n_test"],
            **m,
        })

baseline_df = pd.DataFrame(rows)
baseline_df

**Reading the output.**

- **OLS sits essentially flat** at RMSE $\sim\!33.7$–$34.1\,\mathrm{m^3/ha}$ across the scaling grid, $R^2 \approx 0.38$. More data is not going to help any *linear* model on this feature set — that's the linear point-accuracy ceiling.
- **GBM starts *worse* than OLS at `n_cells=5`** (RMSE $34.3$ vs $33.9$, $R^2$ $0.36$ vs $0.38$) — boosting overfits with only ~3 400 training pixels. From `n_cells=25` onwards GBM overtakes OLS and keeps improving as $N$ grows, ending at RMSE $31.7$ / $R^2 \approx 0.45$ at full $N$. The OLS → GBM gap at full $N$ is roughly **$2\,\mathrm{m^3/ha}$ in RMSE / $0.07$ in $R^2$** — the budget any §7 Bayesian model has to spend if it wants to compete on raw point accuracy alone.
- **Moran's I:** OLS holds at $\sim\!0.18$ across the grid; GBM at $\sim\!0.13$. Trees pick up *some* spatial structure implicitly (forest stands form coherent feature blocks), but residual spatial autocorrelation is still strongly significant ($p = 0.001$) for both methods. Neither model has an explicit spatial component, and the §7 spatial / hierarchical models still have something real to absorb.
- **Bias is small** for both methods at every step, so the standardisation and rescaling round-trip is correct.

The story for §7 is then: a covariate-only BLR (§7.1) should match OLS in RMSE and add credible intervals around it. Hierarchical, spatial-lag, SAR, BNN and GP each have to justify themselves by closing some part of the OLS → GBM gap, dropping Moran's I, or improving calibration over a single point estimate — ideally several at once.

## 7. Bayesian models — building up from simple to complex

This is the comparison the report's *Models* section discusses. Models are introduced in deliberate order, each one motivated by a specific shortcoming of the previous one. Every model uses `slice_step(n_cells)` for its data and `evaluate(...)` for its scoring, so differences between models are not confounded by differences in preprocessing.

### Story arc

| § | Model | What it adds over the previous | What it tests |
|---|---|---|---|
| 7.1 | **BLR via VI** (Pyro `AutoMultivariateNormal`) | Same linear-Gaussian likelihood as the §6 OLS baseline, but with a Gaussian prior on $\beta$ and a half-Cauchy prior on $\sigma$, fitted as a full posterior instead of a point estimate. | Does VI converge on this likelihood / prior? ELBO trace should be monotone-ish. The posterior mean should match OLS's $\hat\beta$ closely (large $N/p$, weak prior), but with credible intervals around it. |
| 7.2 | **BLR via MCMC** (Pyro NUTS) | Asymptotically exact posterior instead of a Gaussian VI approximation. | Inference-method agreement: VI and MCMC should give similar posterior means and intervals on the same model. Disagreement ⟹ VI's mean-field assumption is biting. Required by the course brief. |
| 7.3 | **Heteroscedastic linear** | Lets the observation variance depend on $\mathbf{X}$, $\sigma^2(\mathbf{X})$, instead of being constant. | Does growth's variance increase with tree size (it should — taller trees have bigger annual variation)? |
| 7.4 | **Hierarchical (random intercept per BK)** | Partial pooling per indexruta cell — admits that some 500 m blocks just grow faster than others for reasons outside our covariates. | Does the random-intercept variance dominate the residual variance? If so, the per-BK story is doing real work. |
| 7.5 | **Spatial-lag covariates** | Augments each row with the mean of its rook neighbours' features. | Does explicit local context help over a per-cell intercept? |
| 7.6 | **SAR ($y = \rho W y + X \beta + \varepsilon$)** | Spatial autoregression of the response on neighbour predictions. The Bayesian Spatial Count Models example's correlated noise term, applied to the mean function rather than the noise. | Should drop Moran's I substantially if it's working. |
| 7.7 | **BNN with mean-field SVI** | Nonlinear $f(\mathbf{X})$ instead of $\beta^T \mathbf{X}$. Bayesian neural net, two hidden layers. | Does nonlinearity matter for this feature set? Or does the relationship really look linear? Headline comparison: BNN should at least approach §6 GBM's RMSE. |
| 7.8 | **GP variants** (exact GP at small `n_cells`, SVGP at full) | Nonparametric prior over functions with built-in length scales for spatial smoothness. | Direct Bayesian alternative to SAR for spatial structure; ARD lengthscales tell us which features matter. |

Each subsection follows the same template:

1. *Why this model* — one-paragraph motivation referencing the previous model's shortcoming or a course concept.
2. *Model definition* — Pyro `model()` and `guide()` (or MCMC kernel) with priors named and briefly justified.
3. *Fit + diagnostics* — ELBO trace for VI, $\hat{R}$ + effective sample size for MCMC.
4. *Predict + evaluate* — through `slice_step` / `evaluate`, results appended to a comparison table.
5. *Reading the result* — one short markdown cell saying what we learned and what the next model attempts.

Models land here as separate sub-sections rather than all in one cell so a reviewer can follow the comparison without having to scroll between definitions and results.

## 8. Diagnostics and sanity checks

The course brief flags three diagnostics that any Bayesian-modelling project should make visible. We collect them here so the reviewer can find them in one place.

1. **Generative recovery.** For at least one model (the BLR), draw synthetic data $\tilde{y} \sim p(y \mid \mathbf{X}, \theta^{\star})$ from known parameters $\theta^{\star}$, run inference, and confirm the posterior covers $\theta^{\star}$. If it doesn't, either the model is mis-specified or inference is broken — and we need to know before reading any real-data results.
2. **Inference-method agreement.** §7.1 (VI) and §7.2 (MCMC) fit the *same* model with different inference algorithms. Posterior means and 90% credible intervals on $\beta$ should agree to within a few percent of $y$'s standard deviation; a systematic disagreement diagnoses VI's mean-field bias.
3. **Posterior predictive checks.** For the best-performing model, draw posterior predictive samples and compare summary statistics ($y$'s mean, variance, quantiles, spatial autocorrelation of replicated data vs. observed) to confirm the fitted model produces data that *looks like* the training data. A failed PPC indicates a structural model misfit even when point-prediction RMSE is competitive.

Implementations land in this section as separate cells once the §7 models are in place. The order matters: PPC and inference-method agreement only make sense once the models exist; generative recovery can be done first as a unit test of the BLR machinery.